# Task 05 — Implicit ALS

This notebook is a read-only analytical view of the immutable `task05_implicit_als_v1` artifact. It uses the production restore and validation APIs and never refits the model or reads validation events into training.

In [ ]:
import json
from pathlib import Path

import polars as pl

from implicit_model import ImplicitALSModel
from interfaces import FINAL_RECOMMENDATION_SCHEMA
from validation import validate_final_recommendations

artifact_dir = Path("artifacts/task05_implicit_als_v1")
metrics = json.loads((artifact_dir / "metrics.json").read_text(encoding="utf-8"))
resolved = json.loads((artifact_dir / "config.json").read_text(encoding="utf-8"))
portable = json.loads((artifact_dir / "model_config.json").read_text(encoding="utf-8"))
recommendations = pl.read_parquet(artifact_dir / "recommendations.parquet")
model = ImplicitALSModel.from_artifact(artifact_dir)

In [ ]:
stage_winners = pl.DataFrame(
    [
        {
            "stage": stage["stage"],
            "changed_block": stage["changed_block"],
            "winner_config_id": stage["winner_config_id"],
            **stage["summary"][stage["winner_config_id"]],
        }
        for stage in metrics["selection_stages"]
    ]
).select(
    "stage",
    "changed_block",
    "winner_config_id",
    "mean_union_oracle_p20_all_targets_gain",
    "mean_candidate_oracle_p20_all_targets",
    "mean_precision_at_20_all_targets",
    "mean_runtime_seconds",
)
display(stage_winners)

In [ ]:
selected_id = metrics["selected_config_id"]
selected_folds = pl.DataFrame(
    [
        {"cutoff": fold["fold"]["cutoff"], **result}
        for fold in metrics["selection_stages"][-1]["folds"]
        for result in fold["configs"]
        if result["config_id"] == selected_id
    ]
).select(
    "cutoff",
    "precision_at_20_all_targets",
    "candidate_recall",
    "candidate_oracle_p20_all_targets",
    "union_oracle_p20_all_targets_gain",
    "exclusive_hits",
)
display(selected_folds)

In [ ]:
task04_metrics = json.loads(Path("artifacts/task04_item2item_v1/metrics.json").read_text(encoding="utf-8"))
canonical_comparison = pl.DataFrame(
    [
        {"run": "task04_item2item", **{key: task04_metrics[key] for key in ("precision_at_20_all_targets", "precision_at_20_labeled_users", "candidate_recall", "candidate_oracle_p20_all_targets", "final_hits")}},
        {"run": "task05_implicit_als", **{key: metrics[key] for key in ("precision_at_20_all_targets", "precision_at_20_labeled_users", "candidate_recall", "candidate_oracle_p20_all_targets", "final_hits")}},
    ]
)
display(canonical_comparison)
display(
    pl.DataFrame(
        [
            {"source_set": "task02+task03+task04", "candidate_recall": metrics["canonical_fixed_union"]["baseline_union_candidate_recall"], "oracle_p20_all": metrics["canonical_fixed_union"]["baseline_union_candidate_oracle_p20_all_targets"]},
            {"source_set": "fixed_union+ALS", "candidate_recall": metrics["union_candidate_recall"], "oracle_p20_all": metrics["union_candidate_oracle_p20_all_targets"]},
        ]
    )
)

In [ ]:
assert recommendations.schema == FINAL_RECOMMENDATION_SCHEMA
validate_final_recommendations(recommendations, expected_k=20)
assert metrics["canonical_evaluated_config_count"] == 1
assert metrics["deterministic_recommendations_match"]
assert metrics["artifact_restore_tolerance"]["observed_user_factor_max_abs_diff"] == 0.0
assert metrics["artifact_restore_tolerance"]["observed_item_factor_max_abs_diff"] == 0.0
artifact_checks = {
    "selected_config_id": model.config.config_id,
    "recommendation_users": recommendations.height,
    "min_items_per_user": recommendations.select(pl.col("item_ids").list.len().min()).item(),
    "max_items_per_user": recommendations.select(pl.col("item_ids").list.len().max()).item(),
    "user_factors_shape": model.backend.user_factors.shape,
    "item_factors_shape": model.backend.item_factors.shape,
    "factor_dtype": str(model.backend.item_factors.dtype),
    "fit_history_sha256": portable["metadata"]["fit_history_sha256"],
}
artifact_checks

## Interpretation

The rolling-only winner uses event-strength confidence, no temporal decay, 128 factors, regularization 0.1, and 15 iterations. Its positive mean rolling gain and non-zero exclusive hits confirm complementary value over the frozen task02+task03+task04 sources. The winner was then fitted and evaluated exactly once on canonical. ALS improves standalone canonical Precision@20 over task04 and becomes the current best run, while the larger fixed-union+ALS oracle gain motivates task06 source-aware candidate union and ranking.